# 03 — Audio Processor
Generates narration with Piper. Background music, ambience and SFX are supplied through Drive libraries and are mixed by the video stage.

In [ ]:
import os,sys,subprocess
ROOT="/content/black-history-factory"
REPO_URL="https://github.com/jonbBla/black-history-factory.git"  # put your GitHub repo URL here if you want automatic git clone
if not os.path.exists(ROOT):
    if not REPO_URL: raise RuntimeError("Set REPO_URL to your GitHub repository before running.")
    subprocess.run(["git","clone",REPO_URL,ROOT],check=True)
sys.path.insert(0,ROOT)
from factory.drive import mount_drive,DrivePaths
from factory.config import Config
MYDRIVE=mount_drive(); paths=DrivePaths(os.path.join(MYDRIVE,"BLACK_HISTORY_FACTORY")); paths.ensure_tree(); config=Config.load(paths.root)
print(paths.root)


In [ ]:
from factory.audio_engine import load_piper_voice,run
from factory.utils import read_json,write_json_atomic
from factory import status
PIPER_MODEL="/content/en_US-lessac-medium.onnx"  # change to your voice model
if not os.path.exists(PIPER_MODEL): raise FileNotFoundError("Upload a Piper .onnx voice and set PIPER_MODEL")
voice=load_piper_voice(PIPER_MODEL)
while True:
    jobs=[j for j in os.listdir(paths("02_JOBS")) if (read_json(paths.manifest(j),{}) or {}).get("status")=="IMAGES_READY"]
    if not jobs: print("No audio jobs available."); break
    for job_id in jobs:
        try:
            scenes=read_json(paths.scenes(job_id),[]); status.set_processor(paths,"audio","running",job_id,"narration")
            run(paths,job_id,scenes,voice,config)
            d=read_json(paths.manifest(job_id),{}); d["status"]="AUDIO_READY"; write_json_atomic(paths.manifest(job_id),d)
            status.set_processor(paths,"audio","idle",job_id,"ready"); print("AUDIO READY",job_id)
        except Exception as e:
            d=read_json(paths.manifest(job_id),{}); d.update(audio_error=str(e)); write_json_atomic(paths.manifest(job_id),d); status.set_processor(paths,"audio","error",job_id,str(e))
